### Chatbot for natural disaster

Model that can be used :
microsoft/Phi-3-mini-4k-instruct (dataset from between May and June 2024)   quick answer
microsoft/Phi-4-mini-instruct (Trained between November and December 2024)  slow answer

If there is a gpu and cuda installed, set device_map = "cuda", otherwise set device_map = "cpu"

In [ ]:
import requests
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import gradio as gr
import os
# tokenizer and initialisation of the model
#microsoft/Phi-4-mini-instruct
model_id = "microsoft/Phi-3-mini-4k-instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

import feedparser

import feedparser

import feedparser
from datetime import datetime

def get_latest_disaster_news(user_input, max_articles=5):
    """
    Recherche d’actualités sur les catastrophes naturelles en lien avec la question.
    Renvoie un résumé formaté avec titre, lien et date.
    """
    query = user_input.replace(" ", "+")
    rss_url = f"https://news.google.com/rss/search?q={query}&hl=en&gl=US&ceid=US:en"

    feed = feedparser.parse(rss_url)

    if not feed.entries:
        return "No recent news found related to your question."

    # Mots-clés pour filtrer les événements naturels
    keywords = [
        "earthquake", "flood", "wildfire", "hurricane", "tornado", "tsunami",
        "volcano", "landslide", "natural disaster", "storm", "cyclone", "eruption",
        "mudslide", "seismic", "disaster", "evacuation", "emergency", "aftershock"
    ]

    relevant_articles = []
    for entry in feed.entries:
        title = entry.title
        summary = entry.get("summary", "").lower()
        link = entry.link
        date = entry.get("published_parsed")

        # Formatage de la date
        if date:
            pub_date = datetime(*date[:6]).strftime("%Y-%m-%d %H:%M")
        else:
            pub_date = "Date unknown"

        content = f"{title.lower()} {summary}"
        if any(kw in content for kw in keywords):
            relevant_articles.append(f"- [{title}]({link}) _(Published: {pub_date})_")
            if len(relevant_articles) >= max_articles:
                break

    if not relevant_articles:
        return "No relevant disaster-related news found for your question."

    return "Here are the latest disaster-related news articles related to your question:\n\n" + "\n".join(relevant_articles)




# strict system prompt for natural disaster
system_prompt = (
    "You are an AI assistant specialized strictly in natural disasters. "
    "Answer only questions about natural disasters, safety, preparedness, and recent events. "
    "If the question is unrelated, politely refuse."
)

def chat_with_model(user_input, chat_history=None):
    if chat_history is None:
        chat_history = []

    # 1. Génération initiale du modèle
    messages = [{"role": "system", "content": system_prompt}] + chat_history + [{"role": "user", "content": user_input}]
    chat_input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    chat_input_tokens = tokenizer(chat_input_text, return_tensors="pt").to(model.device)
    input_length = chat_input_tokens["input_ids"].shape[1]
    max_tokens = min(1024, 4096 - input_length)

    outputs = model.generate(
        **chat_input_tokens,
        max_new_tokens=max_tokens,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    generated_tokens = outputs[0][input_length:]
    model_response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    # 2. Détection d'une réponse basée sur des limites de connaissance
    lower_response = model_response.lower()
    triggers = [
        "as of", "my knowledge", "cutoff", "i was trained", "september 2023",
        "i don’t have real-time data", "not up to date", "do not have current", "i'm not updated", "training data ends","I'm sorry,"
    ]
    limited_knowledge = any(trigger in lower_response for trigger in triggers)

    # 3. Recherche seulement si la réponse est affectée par la limite de données
    if limited_knowledge:
        news_summary = get_latest_disaster_news(user_input)
        full_response = (
            f"{model_response}\n\n"
            f"🔎 Since my training data might be outdated, here are some recent news articles related to your question:\n\n"
            f"{news_summary}"
        )
    else:
        full_response = model_response

    # 4. Mise à jour de l’historique
    chat_history.append({"role": "user", "content": user_input})
    chat_history.append({"role": "assistant", "content": full_response})

    return chat_history, chat_history

with gr.Blocks() as demo:
    chatbot_1 = gr.Chatbot(label="Natural disasters assistant", type="messages", height=560)
    msg = gr.Textbox(label="Pose ta question")
    state = gr.State([])

    with gr.Row():
        send_btn = gr.Button("Envoyer")
        reset_btn = gr.Button("Réinitialiser le chatbot", variant="stop")

    # Fonction de reset
    def reset_chat():
        return [], [], ""

    # Envoi message
    send_btn.click(chat_with_model, inputs=[msg, state], outputs=[chatbot_1, state])
    send_btn.click(lambda: "", None, msg)

    # Reset chatbot
    reset_btn.click(reset_chat, outputs=[chatbot_1, state, msg])

demo.launch(server_name="0.0.0.0", server_port=7860,share=True)

c:\Users\noxra\KMUTNB_internship\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:12<00:00,  6.12s/it]


* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://761d1082bd75793e67.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Chatbot for electrical fault

In [1]:
import requests
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import gradio as gr
import os
# tokenizer and initialisation of the model
#microsoft/Phi-4-mini-instruct
model_id = "microsoft/Phi-3-mini-4k-instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

import feedparser

from datetime import datetime

def get_latest_disaster_news(user_input, max_articles=5):

    query = user_input.replace(" ", "+")
    rss_url = f"https://news.google.com/rss/search?q={query}&hl=en&gl=US&ceid=US:en"

    feed = feedparser.parse(rss_url)

    if not feed.entries:
        return "No recent news found related to your question."

    # Mots-clés pour filtrer les événements naturels

    keywords = [
    "electrical component",
    "printed circuit board (PCB)",
    "embedded system",
    "sensors",
    "passive components",
    "active components",
    "electric machine",

    "early failure",
    "material fatigue",
    "useful life",
    "residual life",
    "failure modes",
    "aging mechanisms",
    "electrical anomaly",
    "short circuit",
    "overheating",
    "electrical arcing",
    "performance drift",

    "failure prediction",
    "reliability model",
    "predictive maintenance",
    "diagnostics",
    "prognostics",
    "RUL (Remaining Useful Life)",
    "probabilistic estimation",
    "machine learning",
    "predictive models",
    "neural networks",
    "time series analysis",

    "condition monitoring",
    "real-time monitoring",
    "temperature sensors",
    "current sensors",
    "voltage sensors",
    "vibration analysis",
    "thermal analysis",
    "spectral analysis",
    "electrical signals",
    "data acquisition",
    "non-intrusive measurement",

    "failure analysis",

]

    relevant_articles = []
    for entry in feed.entries:
        title = entry.title
        summary = entry.get("summary", "").lower()
        link = entry.link
        date = entry.get("published_parsed")

        # Formatage de la date
        if date:
            pub_date = datetime(*date[:6]).strftime("%Y-%m-%d %H:%M")
        else:
            pub_date = "Date unknown"

        content = f"{title.lower()} {summary}"
        if any(kw in content for kw in keywords):
            relevant_articles.append(f"- [{title}]({link}) _(Published: {pub_date})_")
            if len(relevant_articles) >= max_articles:
                break

    if not relevant_articles:
        return "No relevant disaster-related news found for your question."

    return "Here are the latest disaster-related news articles related to your question:\n\n" + "\n".join(relevant_articles)

# strict system prompt for natural disaster
system_prompt = (
    "You are an AI assistant specialized strictly in electrical fault and machine learning. "
    "Answer only questions about fault in electrical system and machine learning"
    "If the question is unrelated, politely refuse."
)

def chat_with_model(user_input, chat_history=None):
    if chat_history is None:
        chat_history = []

    # 1. Génération initiale du modèle
    messages = [{"role": "system", "content": system_prompt}] + chat_history + [{"role": "user", "content": user_input}]
    chat_input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    chat_input_tokens = tokenizer(chat_input_text, return_tensors="pt").to(model.device)
    input_length = chat_input_tokens["input_ids"].shape[1]
    max_tokens = min(1024, 4096 - input_length)

    outputs = model.generate(
        **chat_input_tokens,
        max_new_tokens=max_tokens,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    generated_tokens = outputs[0][input_length:]
    model_response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    # 2. Détection d'une réponse basée sur des limites de connaissance
    lower_response = model_response.lower()
    triggers = [
        "as of", "my knowledge", "cutoff", "i was trained", "september 2023",
        "i don’t have real-time data", "not up to date", "do not have current", "i'm not updated", "training data ends","I'm sorry,"
    ]
    limited_knowledge = any(trigger in lower_response for trigger in triggers)

    # 3. Recherche seulement si la réponse est affectée par la limite de données
    if limited_knowledge:
        news_summary = get_latest_disaster_news(user_input)
        full_response = (
            f"{model_response}\n\n"
            f"🔎 Since my training data might be outdated, here are some recent news articles related to your question:\n\n"
            f"{news_summary}"
        )
    else:
        full_response = model_response

    # 4. Mise à jour de l’historique
    chat_history.append({"role": "user", "content": user_input})
    chat_history.append({"role": "assistant", "content": full_response})

    return chat_history, chat_history

with gr.Blocks() as demo:
    chatbot = gr.Chatbot(label="Electrical fault assistant", type="messages", height=560)
    msg = gr.Textbox(label="Ask questions")
    state = gr.State([])

    with gr.Row():
        send_btn = gr.Button("Send")
        reset_btn = gr.Button("Restart chatbot", variant="stop")

    # Fonction de reset
    def reset_chat():
        initial_state = [{"role": "system", "content": system_prompt}]
        return [], initial_state, ""

    # Envoi message
    send_btn.click(chat_with_model, inputs=[msg, state], outputs=[chatbot, state])
    send_btn.click(lambda: "", None, msg)

    # Reset chatbot
    reset_btn.click(reset_chat, outputs=[chatbot, state, msg])

demo.launch(share=True)

c:\Users\noxra\KMUTNB_internship\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:10<00:00,  5.40s/it]


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://eee52d96bacc8e2689.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## model comparison

In [ ]:
import time
import requests
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import gradio as gr
import os
import pandas as pd
# tokenizer and initialisation of the model
#microsoft/Phi-4-mini-instruct
model_id = "microsoft/Phi-3-mini-4k-instruct"

save = pd.DataFrame(["date","model","question","answer","time","evaluation"])

import feedparser
from datetime import datetime

def get_latest_disaster_news(user_input, max_articles=5):
    """
    Recherche d’actualités sur les catastrophes naturelles en lien avec la question.
    Renvoie un résumé formaté avec titre, lien et date.
    """
    query = user_input.replace(" ", "+")
    rss_url = f"https://news.google.com/rss/search?q={query}&hl=en&gl=US&ceid=US:en"

    feed = feedparser.parse(rss_url)

    if not feed.entries:
        return "No recent news found related to your question."

    # Mots-clés pour filtrer les événements naturels
    keywords = [
    "electrical component",
    "printed circuit board (PCB)",
    "embedded system",
    "sensors",
    "passive components",
    "active components",
    "electric machine",

    "early failure",
    "material fatigue",
    "useful life",
    "residual life",
    "failure modes",
    "aging mechanisms",
    "electrical anomaly",
    "short circuit",
    "overheating",
    "electrical arcing",
    "performance drift",

    "failure prediction",
    "reliability model",
    "predictive maintenance",
    "diagnostics",
    "prognostics",
    "RUL (Remaining Useful Life)",
    "probabilistic estimation",
    "machine learning",
    "predictive models",
    "neural networks",
    "time series analysis",

    "condition monitoring",
    "real-time monitoring",
    "temperature sensors",
    "current sensors",
    "voltage sensors",
    "vibration analysis",
    "thermal analysis",
    "spectral analysis",
    "electrical signals",
    "data acquisition",
    "non-intrusive measurement",

    "failure analysis",

]

    relevant_articles = []
    for entry in feed.entries:
        title = entry.title
        summary = entry.get("summary", "").lower()
        link = entry.link
        date = entry.get("published_parsed")

        # Formatage de la date
        if date:
            pub_date = datetime(*date[:6]).strftime("%Y-%m-%d %H:%M")
        else:
            pub_date = "Date unknown"

        content = f"{title.lower()} {summary}"
        if any(kw in content for kw in keywords):
            relevant_articles.append(f"- [{title}]({link}) _(Published: {pub_date})_")
            if len(relevant_articles) >= max_articles:
                break

    if not relevant_articles:
        return "No relevant disaster-related news found for your question."

    return "Here are the latest disaster-related news articles related to your question:\n\n" + "\n".join(relevant_articles)

# strict system prompt for natural disaster
system_prompt = (
    "You are a precise and concise AI assistant specialized strictly in electrical faults and machine learning."
    " Only answer the specific questions asked by the user."
    " Do not generate or invent additional questions or simulate Q&A by yourself."
    " Keep your answers focused, technical, and without self-dialogue."
    " If the user's question is unrelated to electrical systems or machine learning, politely refuse to answer."
)

def chat_with_model(user_input, model_id, chat_history=None):
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="cuda",
        torch_dtype="auto",
        trust_remote_code=False,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    if chat_history is None:
        chat_history = []

    # SYSTEM + messages
    messages = [{"role": "system", "content": system_prompt}] + chat_history + [{"role": "user", "content": user_input}]
    chat_input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    chat_input_tokens = tokenizer(chat_input_text, return_tensors="pt").to(model.device)
    input_length = chat_input_tokens["input_ids"].shape[1]
    max_tokens = min(1024, 4096 - input_length)

    # Mesure du temps de réponse
    start_time = time.time()

    outputs = model.generate(
        **chat_input_tokens,
        max_new_tokens=max_tokens,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    response_time = round(time.time() - start_time, 2)  # en secondes

    generated_tokens = outputs[0][input_length:]
    model_response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    lower_response = model_response.lower()
    triggers = [
        "as of", "my knowledge", "cutoff", "i was trained", "september 2023",
        "i don’t have real-time data", "not up to date", "do not have current", "i'm not updated", "training data ends","I'm sorry,"
    ]
    limited_knowledge = any(trigger in lower_response for trigger in triggers)

    if limited_knowledge:
        news_summary = get_latest_disaster_news(user_input)
        full_response = (
            f"{model_response}\n\n"
            f"🔎 Since my training data might be outdated, here are some recent news articles related to your question:\n\n"
            f"{news_summary}"
        )
    else:
        full_response = model_response

    chat_history.append({"role": "user", "content": user_input})
    chat_history.append({"role": "assistant", "content": full_response})

    return chat_history, chat_history, response_time

# Fonction de sauvegarde dans le DataFrame + CSV
def save_to_csv(chat_history, model_id, evaluation, response_time):
    global save
    if not chat_history or len(chat_history) < 2:
        return "⚠️ Nothing to save."

    last_user_msg = chat_history[-2]["content"]
    last_response = chat_history[-1]["content"]
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    new_row = pd.DataFrame([[timestamp, model_id, last_user_msg, last_response, response_time, evaluation]],
                           columns=["date", "model", "question", "answer", "time", "evaluation"])
    
    save = pd.concat([save, new_row], ignore_index=True)

    file_path = "chat_log.csv"
    if os.path.exists(file_path):
        existing_df = pd.read_csv(file_path)
        combined_df = pd.concat([existing_df, new_row], ignore_index=True)
        combined_df.to_csv(file_path, index=False)
    else:
        new_row.to_csv(file_path, index=False)

    return "✅ Model response saved to chat_log.csv."

# Interface Gradio
with gr.Blocks() as demo:  
    with gr.Row():
        model_id_box_1 = gr.Dropdown(choices=["microsoft/Phi-3-mini-4k-instruct", "microsoft/Phi-4-mini-instruct", "meta-llama/Llama-3.1-8B-Instruct", "openai-community/gpt2"], value=model_id, visible=True)
        model_id_box_2 = gr.Dropdown(choices=["microsoft/Phi-3-mini-4k-instruct", "microsoft/Phi-4-mini-instruct", "meta-llama/Llama-3.1-8B-Instruct", "openai-community/gpt2"], value=model_id, visible=True)

    with gr.Row():
        chatbot_1 = gr.Chatbot(label="Electrical fault assistant model 1", type="messages", height=500)
        chatbot_2 = gr.Chatbot(label="Electrical fault assistant model 2", type="messages", height=500)

    msg = gr.Textbox(label="Ask questions")
    
    state_1 = gr.State([])
    state_2 = gr.State([])

    time_1 = gr.State(0.0)
    time_2 = gr.State(0.0)

    with gr.Row():
        send_btn = gr.Button("Send")
        reset_btn = gr.Button("Reset chatbot", variant="stop")

    def reset_chat():
        return [], [], "", 0.0

    with gr.Row():
        evaluation_1 = gr.Dropdown(label="Evaluate Model 1", choices=["Excellent", "Good", "Average", "Poor", "Irrelevant"])
        save_btn_1 = gr.Button("Save Model 1 Response")

        evaluation_2 = gr.Dropdown(label="Evaluate Model 2", choices=["Excellent", "Good", "Average", "Poor", "Irrelevant"])
        save_btn_2 = gr.Button("Save Model 2 Response")

    # Click behavior
        # Envoi message

    send_btn.click(chat_with_model, inputs=[msg, model_id_box_1, state_1], outputs=[chatbot_1, state_1, time_1])

    send_btn.click(chat_with_model, inputs=[msg, model_id_box_2, state_2], outputs=[chatbot_2, state_2, time_2])

    send_btn.click(lambda: "", None, msg)

    # Reset chatbot
    reset_btn.click(reset_chat, outputs=[chatbot_1, state_1, msg])
    reset_btn.click(reset_chat, outputs=[chatbot_2, state_2, msg])
    save_btn_1.click(fn=save_to_csv, inputs=[state_1, model_id_box_1, evaluation_1, time_1], outputs=[gr.Textbox(visible=False, label="Status Message")])
    save_btn_2.click(fn=save_to_csv, inputs=[state_2, model_id_box_2, evaluation_2, time_2], outputs=[gr.Textbox(visible=False, label="Status Message")])


demo.launch(share=True)



c:\Users\noxra\KMUTNB_internship\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://e66449bf1e27512c4b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Loading checkpoint shards: 100%|██████████| 2/2 [00:12<00:00,  6.27s/it]


In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import sacrebleu

# Liste de questions à tester
questions = [
    "What should I do during an earthquake?",
    "How can I prepare for a flood?",
    "What causes landslides?",
    "Is a tsunami the same as a tidal wave?",
    "What emergency kit should I have for a hurricane?"
]

# Références humaines (à adapter si tu as des réponses de référence réelles)
reference_answers = [
    "During an earthquake, drop to the ground, take cover under something sturdy, and hold on until the shaking stops.",
    "To prepare for a flood, move valuables to higher ground, have sandbags ready, and create an evacuation plan.",
    "Landslides are caused by heavy rain, earthquakes, volcanic activity, or human interference with the land.",
    "A tsunami is not the same as a tidal wave. Tsunamis are caused by underwater disturbances like earthquakes.",
    "You should have water, non-perishable food, flashlight, batteries, medicine, and important documents ready."
]

# Modèles à comparer
models_info = {
    "model_1": {
        "id": "microsoft/Phi-3-mini-4k-instruct",
        "device": "cpu"
    },
    "model_2": {
        "id": "microsoft/Phi-4-mini-instruct",
        "device": "cuda:0"
    }
}

# Système de prompt
system_prompt = (
    "You are an assistant specialized in natural disasters. "
    "Answer only questions related to natural disasters, preparedness, and safety."
)

# Fonction de génération de réponse
def generate_answer(model, tokenizer, question):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs["input_ids"].shape[1]

    output = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = output[0][input_length:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

# Chargement des modèles
for model_name, info in models_info.items():
    print(f"Loading {model_name}...")
    model = AutoModelForCausalLM.from_pretrained(info["id"], device_map=info["device"], torch_dtype=torch.float16)
    tokenizer = AutoTokenizer.from_pretrained(info["id"])
    models_info[model_name]["model"] = model
    models_info[model_name]["tokenizer"] = tokenizer

# Génération des réponses
for model_name, info in models_info.items():
    print(f"\nGenerating answers for {model_name}...\n")
    model = info["model"]
    tokenizer = info["tokenizer"]
    predictions = []

    for q in questions:
        response = generate_answer(model, tokenizer, q)
        predictions.append(response)
        print(f"Q: {q}\n→ {response}\n")

    # BLEU computation
    bleu = sacrebleu.corpus_bleu(predictions, [reference_answers])
    print(f"\n🔵 BLEU score for {model_name}: {bleu.score:.2f}")


c:\Users\noxra\KMUTNB_internship\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model_1...


Loading checkpoint shards: 100%|██████████| 2/2 [00:19<00:00,  9.86s/it]


Loading model_2...


Loading checkpoint shards: 100%|██████████| 2/2 [00:12<00:00,  6.49s/it]



Generating answers for model_1...

Q: What should I do during an earthquake?
→ During an earthquake, it's crucial to protect yourself by following these steps:


1. Drop to the ground immediately. This is the first and most important action to take when the shaking starts.

2. Take cover by getting under a sturdy piece of furniture or against an interior wall, away from windows and objects that could fall.

3. Hold on to your cover until the shaking stops. Be prepared for aftershocks.

4. Stay indoors until the shaking subsides and it is safe to exit.

5. If you are outdoors, move to a clear area away from buildings, trees, streetlights, and

Q: How can I prepare for a flood?
→ To prepare for a flood, you should take the following steps:


1. **Stay Informed:** Monitor local news and weather reports for flood warnings. Sign up for local alerts and notifications.

2. **Emergency Kit:** Assemble an emergency kit with essentials such as water, non-perishable food, medications, flashlight

In [10]:
from huggingface_hub import snapshot_download
import os

def get_model_size(model_id):
    # Disable symlinks by setting `local_files_only=True` temporarily if needed
    model_dir = snapshot_download(repo_id=model_id, local_dir_use_symlinks=False)
    total_size = 0
    for dirpath, _, filenames in os.walk(model_dir):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_size += os.path.getsize(fp)
    return total_size / (1024 ** 2)

for model_id in ["microsoft/Phi-3-mini-4k-instruct", "microsoft/Phi-4-mini-instruct"]:
    size = get_model_size(model_id)
    print(f"Model {model_id} size on disk: {size:.2f} MB")


Fetching 19 files: 100%|██████████| 19/19 [00:00<?, ?it/s]


Model microsoft/Phi-3-mini-4k-instruct size on disk: 7290.61 MB


Fetching 20 files: 100%|██████████| 20/20 [00:01<00:00, 13.58it/s]

Model microsoft/Phi-4-mini-instruct size on disk: 7337.62 MB


In [ ]:
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

def benchmark_model(model_id, input_text, device="cuda"):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
    model.eval()

    inputs = tokenizer(input_text, return_tensors="pt").to(device)

    # Nettoyer la mémoire GPU avant test
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device)

    # Warm-up (pour compilation etc.)
    with torch.no_grad():
        _ = model.generate(**inputs, max_new_tokens=50)

    # Mesure mémoire et temps
    torch.cuda.reset_peak_memory_stats(device)
    start = time.time()
    with torch.no_grad():
        _ = model.generate(**inputs, max_new_tokens=50)
    end = time.time()

    peak_mem = torch.cuda.max_memory_allocated(device) / (1024**2)  # Mo
    duration = end - start

    print(f"Model {model_id} - Peak memory usage: {peak_mem:.2f} MB, Inference time: {duration:.2f} s")

# Exemple d'input
prompt = "What should I do in case of an earthquake?"

for model_id in ["microsoft/Phi-3-mini-4k-instruct"]:
    benchmark_model(model_id, prompt)
